In [1]:
!pip install transformers==4.55.4 datasets evaluate rouge_score sentencepiece accelerate -q

In [2]:
import numpy as np
import evaluate
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "parquet",
    data_files={
        "train": "xlsum-train.parquet",
        "validation": "xlsum-validation.parquet",
        "test": "xlsum-test.parquet",
    }
)

train_dataset = dataset["train"].shuffle(seed=42).select(range(3000))
val_dataset = dataset["validation"].shuffle(seed=42).select(range(500))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(dataset)
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 38242
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 4780
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 4780
    })
})
Train samples: 3000
Validation samples: 500
Test samples: 500


In [4]:
model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Using device:", device)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
c:\Users\USER\Documents\NLP_Project_mT5\venv\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Using device: cuda


In [5]:
max_input_length = 512
max_target_length = 64

def preprocess_function(examples):
    inputs = ["summarize: " + str(text) for text in examples["text"]]
    targets = [str(summary) for summary in examples["summary"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

In [6]:
sample = tokenized_train[0]

print("INPUT:")
print(tokenizer.decode(sample["input_ids"], skip_special_tokens=True))

print("\nLABEL:")
print(tokenizer.decode(sample["labels"], skip_special_tokens=True))

INPUT:
summarize: Bulan Januari lalu, militan ISIS membebaskan anggota sekte Yazidi yang mereka tahan. Diketahui bahwa mereka yang dilepaskan adalah orang-orang yang sudah berusia lanjut atau dalam keadaan sakit. Komandan pasukan Kurdi, Westa Rasul, menyatakan beberapa di antara yang dilepaskan adalah 'wanita dan anak-anak'. Puluhan ribu penganut sekte Yazidi dipaksa untuk mengungsi ketika ISIS menyerbu desa-desa mereka pada bulan Agustus tahun lalu. Ribuan dari mereka mengungsi hingga ke gunung-gunung di Irak utara. Kelompok militan ini mengutuk keimanan sekte Yazidi dan membunuh ratusan anggota komunitas ini. Perserikatan Bangsa-bangsa menyatakan kemungkinan bahwa ISIS melakukan pemusnahan massal atau genosida terhadap sekte Yazidi. Ribuan lain anggota sekte ini ditangkap dan ditahan. Kebanyakan dari mereka adalah perempuan yang dilaporkan dijual untuk dijadikan budak seks. Bulan Januari lalu, militan ISIS juga membebaskan pengikut sekte Yazidi dari tahanan mereka.

LABEL:
Kelompok m

In [7]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

In [8]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Fix invalid token IDs before decoding
    predictions = np.where(
        predictions < 0,
        tokenizer.pad_token_id,
        predictions
    )

    predictions = np.where(
        predictions >= tokenizer.vocab_size,
        tokenizer.pad_token_id,
        predictions
    )

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    result = rouge.compute(
    predictions=decoded_preds,
    references=decoded_labels,
    use_stemmer=True
    )

    return {
       "rouge1": round(result["rouge1"] * 100, 4),
        "rouge2": round(result["rouge2"] * 100, 4),
        "rougeL": round(result["rougeL"] * 100, 4),
        "rougeLsum": round(result["rougeLsum"] * 100, 4)
    }

In [9]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_xlsum_results",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=3e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,

    predict_with_generate=True,
    generation_max_length=64,
    generation_num_beams=4,

    logging_dir="./logs",
    logging_steps=50,
    report_to="none",

    fp16=False
)

In [10]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

C:\Users\USER\AppData\Local\Temp\ipykernel_19052\3226501058.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,5.156900,3.333283,13.475900,4.320100,12.022600,12.044300
2,4.524100,3.186215,16.473900,5.542600,14.204100,14.242000
3,4.332600,3.147020,17.589700,6.026500,15.045300,15.064600


TrainOutput(global_step=1125, training_loss=6.058015896267361, metrics={'train_runtime': 516.4433, 'train_samples_per_second': 17.427, 'train_steps_per_second': 2.178, 'total_flos': 4475031700193280.0, 'train_loss': 6.058015896267361, 'epoch': 3.0})

In [12]:
test_results = trainer.evaluate(eval_dataset=tokenized_test)

print("Test Results:")
test_results

Test Results:


{'eval_loss': 3.1493794918060303,
 'eval_rouge1': 17.3535,
 'eval_rouge2': 5.9361,
 'eval_rougeL': 14.8419,
 'eval_rougeLsum': 14.8848,
 'eval_runtime': 72.6059,
 'eval_samples_per_second': 6.886,
 'eval_steps_per_second': 3.443,
 'epoch': 3.0}

In [13]:
def generate_summary(text):
    input_text = "summarize: " + str(text)

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

In [14]:
sample_article = test_dataset[0]["text"]
reference_summary = test_dataset[0]["summary"]
generated_summary = generate_summary(sample_article)

print("ARTICLE:")
print(sample_article)

print("\nREFERENCE SUMMARY:")
print(reference_summary)

print("\nGENERATED SUMMARY:")
print(generated_summary)

ARTICLE:
Pandemi Covid-19 telah meningkatkan minat berinvestasi emas. Kenaikan harga itu salah satu penyebab utamanya karena permainan para pedagang emas dan permintaan yang tinggi, tapi di balik itu, muncul pertanyaan tentang berapa sisa pasokan logam mulia itu di bumi dan kapan akan habis. Emas menjadi buruan masyarakat karena dapat dijadikan sebagai investasi, simbol status ekonomi, dan komponen utama produk elektronik. Tapi jumlah emas di dunia terbatas, dan pada akhirnya akan datang satu saat ketika tidak ada lagi emas yang tersisa untuk ditambang. Puncak produksi emas Beberapa ahli berbicara tentang apakah dunia telah mencapai puncak produksi emas - diukur dengan jumlah terbanyak emas yang pernah ditambang dalam satu tahun - atau tidak. Hal itu ditunjukan dengan mulai menurunnya tren produksi emas dunia. Contohnya, pada tahun 2019, produksi tambang emas dunia turun 1% menjadi 3.531 ton dibandingkan tahun 2018, menurut Dewan Emas Dunia. Penurunan produksi tahunan ini yang pertama 

In [15]:
model.save_pretrained("./mt5_xlsum_indonesian_model")
tokenizer.save_pretrained("./mt5_xlsum_indonesian_model")

print("Model saved successfully.")

Model saved successfully.


In [16]:
loaded_tokenizer = AutoTokenizer.from_pretrained("./mt5_xlsum_indonesian_model")
loaded_model = AutoModelForSeq2SeqLM.from_pretrained("./mt5_xlsum_indonesian_model")

loaded_model.to(device)

print("Saved model loaded successfully.")

Saved model loaded successfully.
